# Pipeline 04Ge: whole-model global explanation (G2b, exploratory)

Where 04Gb/04Gc/04Gd ask about **one feature per call**, this notebook asks the LLM to
describe the **whole model** (all 9 features) in a single call, under **five conditions**
(× 2 XAI models = 10 calls total). It is the additive, exploratory extension of the
revision plan (Phase **G2b**) and opens the two comparisons the per-feature track cannot:

| Axis | Comparison | Conditions |
| --- | --- | --- |
| **1 — Representation** | all 9 curves/plots vs the single beeswarm | `*_all` vs `*_beeswarm` — scored **per GT field**, beeswarm only on direction/rank |
| **2 — Mechanism (push vs pull)** | information pushed in vs pulled via tools | `vision_all`/`json_all` vs `tooluse_all` |
| **Beeswarm readability** | swarm image vs info-matched numbers | `vision_beeswarm` vs `json_beeswarm` — isolates the pure modality effect |

The `json_beeswarm` payload is **deliberately info-matched** to what the swarm image
conveys (ranking + colour direction + coarse spread, **no per-value curve**), so the
vision-vs-json beeswarm difference measures modality, not information (plan G2b Auflage).

**Forced schema → G3 scoreability.** Each answer must emit a rigid
`[FEATURE: name] [EFFECT] … [IMPORTANCE] …` block per feature, then one whole-model
`[RECOMMENDATION]`. `split_whole_model_record` cuts this into per-feature records the
existing G3 rubric + reference judge score unchanged; a feature the answer **omits** is
scored as a miss — the meeting's "only 5 of 9 right" concern, now measurable.

Outputs: raw records `results/global_whole/{condition}_{model}.json`; per-feature split
records `results/global_whole_split/{condition}_{model}_{feature}.json`. Resumable.

> **`RUN_API` guard.** Default `False`: the notebook verifies the whole non-API path
> (prompt assembly, payloads, splitter, coverage, rubric) with a deterministic **stub**
> and writes **nothing** to `results/`. Set `True` for the single billed run.

In [5]:
from __future__ import annotations

import sys, json, time, tempfile
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from utils import (
    EXPLANATIONS_DIR, RESULTS_DIR, PROMPTS_DIR,
    WHOLE_RESULTS_SUBDIR, WHOLE_SPLIT_SUBDIR, WHOLE_CONDITIONS,
    list_global_features, feature_importance_map, describe_curve,
    build_whole_json_all_payload, build_whole_json_beeswarm_payload, whole_plot_paths,
    assemble_whole_system_prompt, build_whole_record, run_resumable_whole_generation,
    write_split_records, split_whole_model_record,
    GlobalToolBox, run_global_tool_use_loop,
)
from utils.llm import (
    ask_text, ask_with_images, _get_client, DEFAULT_MODEL, strip_scratchpad,
)
from utils import rubric

# --- config -----------------------------------------------------------------
RUN_API    = True              # <- set True for the single billed run (10 calls)
LOSS_KEY   = 'poisson_log'
MODEL      = DEFAULT_MODEL
MAX_TOKENS = 4096               # whole-model answer = 9 feature blocks, needs headroom
XAI_MODELS = ['xgb', 'ebm']

PLOTS_DIR = EXPLANATIONS_DIR / 'plots' / 'global'
OUT_DIR   = RESULTS_DIR / WHOLE_RESULTS_SUBDIR
SPLIT_DIR = RESULTS_DIR / WHOLE_SPLIT_SUBDIR

FEATURES   = list_global_features('ebm', explanations_dir=EXPLANATIONS_DIR)
N_FEATURES = len(FEATURES)

print(f'LLM model:  {MODEL}')
print(f'Conditions: {[c.name for c in WHOLE_CONDITIONS]}')
print(f'Calls:      {len(WHOLE_CONDITIONS)} conditions x {len(XAI_MODELS)} models = '
      f'{len(WHOLE_CONDITIONS) * len(XAI_MODELS)} whole-model calls')
print(f'RUN_API:    {RUN_API}  ->  ' + ('BILLED real run' if RUN_API
      else 'stub verification, nothing written to results/'))

LLM model:  claude-sonnet-4-6
Conditions: ['json_all', 'json_beeswarm', 'vision_all', 'vision_beeswarm', 'tooluse_all']
Calls:      5 conditions x 2 models = 10 whole-model calls
RUN_API:    True  ->  BILLED real run


## 1. System prompts + payload preview

One system prompt per `(condition, model)`: byte-identical core + the condition's
handover block, with `{{MODEL}}` and the exact `{{FEATURE_LIST}}` filled. Below we also
preview the info-matched `json_beeswarm` payload to show it carries **only** rank +
colour direction + spread (no curve).

In [6]:
# System prompt per (condition, model). Cached per key at call time (cache_system=True).
SYSTEM = {
    (c.name, m): assemble_whole_system_prompt(c.name, m, prompts_dir=PROMPTS_DIR,
                                              features=FEATURES)
    for c in WHOLE_CONDITIONS for m in XAI_MODELS
}
print('--- vision_beeswarm / ebm system prompt (first 700 chars) ---')
print(SYSTEM[('vision_beeswarm', 'ebm')][:700], '\n...')

print('\n--- json_beeswarm payload (ebm) : info-matched, no curve ---')
bee = build_whole_json_beeswarm_payload('ebm', explanations_dir=EXPLANATIONS_DIR)
print(json.dumps(bee['features'][:3], indent=2), '\n... (', bee['n_features'], 'features )')

print('\n--- json_all payload (ebm) : full curves ---')
allp = build_whole_json_all_payload('ebm', explanations_dir=EXPLANATIONS_DIR)
f0 = allp['features'][0]
print(f"feature={f0['feature']} rank={f0['rank']} curve points={len(f0['curve']['x'])}")

--- vision_beeswarm / ebm system prompt (first 700 chars) ---
You are an expert in explainable AI (XAI). You describe, for staff of a bike rental
company with no technical background, how a prediction model uses **each of its
features** to predict demand. You describe the **whole model** in one pass, one feature
at a time.

## DOMAIN CONTEXT

The Capital Bikeshare system in Washington D.C. rents bikes by the hour. A **EBM**
model predicts how many bikes (`cnt`) are rented in a given hour. Every statement you
make is about this EBM model. It was trained with Poisson deviance loss, so a
feature's contribution is expressed in **log space**: a positive contribution multiplies
the predicted demand up, a negative one multiplies it down. You describe each fea 
...

--- json_beeswarm payload (ebm) : info-matched, no curve ---
[
  {
    "feature": "hr",
    "rank": 1,
    "colour_direction": "varies by category (no ordinal colour trend)",
    "spread": "wide"
  },
  {
    "feature": "temp",
   

## 2. Generate (or stub-verify)

`generate_api` dispatches on the condition's modality: JSON payload (`ask_text`), images
(`ask_with_images`, all-plots or the single swarm), or the tool-use pull loop. When
`RUN_API=False` we build deterministic **stub** records instead (correct ranks + curve
direction), so the split/coverage/scoring path is verified end-to-end at zero cost.

In [7]:
USER_MSG = (
    'Describe how this model uses each of its features to predict hourly bike demand. '
    'Cover every feature with its own block, then give one overall recommendation.'
)

def generate_api(model_name, cond):
    system = SYSTEM[(cond.name, model_name)]
    t0 = time.time()
    extra = None
    try:
        if cond.modality == 'json':
            payload = (build_whole_json_all_payload(model_name, explanations_dir=EXPLANATIONS_DIR)
                       if cond.representation == 'all'
                       else build_whole_json_beeswarm_payload(model_name, explanations_dir=EXPLANATIONS_DIR))
            resp = ask_text(USER_MSG + '\n\n' + json.dumps(payload, indent=2),
                            system=system, model=MODEL, max_tokens=MAX_TOKENS, cache_system=True)
            text, usage = strip_scratchpad(resp['content'][0]['text']), resp.get('usage', {})
        elif cond.modality == 'vision':
            paths = whole_plot_paths(model_name, cond.representation, plots_dir=PLOTS_DIR,
                                     explanations_dir=EXPLANATIONS_DIR)
            resp = ask_with_images(USER_MSG, paths, system=system, model=MODEL,
                                   max_tokens=MAX_TOKENS, cache_system=True)
            text, usage = strip_scratchpad(resp['content'][0]['text']), resp.get('usage', {})
            extra = {'plot_files': [p.name for p in paths]}
        else:  # tooluse (pull)
            toolbox = GlobalToolBox(model_name, explanations_dir=EXPLANATIONS_DIR,
                                    plots_dir=PLOTS_DIR, loss_key=LOSS_KEY,
                                    aggregate_curves=True)
            text, call_log, in_tok, out_tok, stop = run_global_tool_use_loop(
                client, toolbox, user_message=USER_MSG, system=system, model=MODEL,
                max_tokens=MAX_TOKENS, max_rounds=20)
            text, usage = strip_scratchpad(text), {'input_tokens': in_tok, 'output_tokens': out_tok}
            extra = {'stop_reason': stop, 'n_tool_calls': len(call_log), 'tool_calls': call_log}
    except Exception as e:
        print(f'  [ERROR] {cond.name} {model_name}: {type(e).__name__}: {e} -> skip')
        return None
    rec = build_whole_record(condition=cond, model_name=model_name, explanation=text,
                             usage=usage, llm_model=MODEL, loss_key=LOSS_KEY,
                             elapsed_s=round(time.time() - t0, 2),
                             include_cache=(cond.modality != 'tooluse'), extra=extra)
    covered = sum(1 for p in split_whole_model_record(rec, features=FEATURES) if not p['dropped'])
    print(f"  {cond.name:16} {model_name.upper():4} covered={covered}/{N_FEATURES} "
          f"in={rec['usage']['input_tokens']} out={rec['usage']['output_tokens']}")
    return rec


def _stub_explanation(model_name):
    '''Deterministic whole-model answer for the non-API path: valid forced schema,
    correct ranks + curve direction. vision_beeswarm drops the least-important feature
    to exercise the coverage (<9) measurement.'''
    imp = feature_importance_map(model_name, explanations_dir=EXPLANATIONS_DIR)
    lines = ['<analysis>stub</analysis>']
    for f in FEATURES:
        d = describe_curve(model_name, f, explanations_dir=EXPLANATIONS_DIR)
        lines += [f'[FEATURE: {f}]',
                  f'[EFFECT] The effect is {d["direction"]} and {d["monotonicity"]}.',
                  f'[IMPORTANCE] Rank {imp[f]["rank"]} of {N_FEATURES}.']
    lines.append('[RECOMMENDATION] Plan bikes and staff around the top-ranked drivers.')
    return '\n'.join(lines)

def generate_stub(model_name, cond):
    text = _stub_explanation(model_name)
    if cond.name == 'vision_beeswarm':                     # drop the last feature block
        text = text.rsplit(f'[FEATURE: {FEATURES[-1]}]', 1)[0].rstrip() + \
               '\n[RECOMMENDATION] Plan bikes and staff around the top-ranked drivers.'
    rec = build_whole_record(condition=cond, model_name=model_name, explanation=text,
                             usage={'input_tokens': 0, 'output_tokens': 0},
                             llm_model='stub', loss_key=LOSS_KEY)
    covered = sum(1 for p in split_whole_model_record(rec, features=FEATURES) if not p['dropped'])
    print(f'  [stub] {cond.name:16} {model_name.upper():4} covered={covered}/{N_FEATURES}')
    return rec


client = _get_client() if RUN_API else None
if RUN_API:
    records = run_resumable_whole_generation(
        model_names=XAI_MODELS, conditions=WHOLE_CONDITIONS,
        out_dir=OUT_DIR, generate=generate_api)
else:
    print('RUN_API=False -> stub verification (no API calls, nothing written to results/)')
    records = [generate_stub(m, c) for m in XAI_MODELS for c in WHOLE_CONDITIONS]
print(f'\n{len(records)} whole-model record(s).')

  json_all         XGB  covered=9/9 in=3663 out=2433
  json_beeswarm    XGB  covered=9/9 in=588 out=2173
  vision_all       XGB  covered=6/9 in=5301 out=4096
  vision_beeswarm  XGB  covered=4/9 in=811 out=4096
    [1] get_target_overview([]) -> ok
    [1] get_feature_importances([]) -> ok
    [1] get_beeswarm_plot([]) -> ok
    [2] get_feature_curve(['feature']) -> ok
    [2] get_feature_curve(['feature']) -> ok
    [2] get_feature_curve(['feature']) -> ok
    [2] get_feature_curve(['feature']) -> ok
    [2] get_feature_curve(['feature']) -> ok
    [2] get_feature_curve(['feature']) -> ok
    [2] get_feature_curve(['feature']) -> ok
    [2] get_feature_curve(['feature']) -> ok
    [2] get_feature_curve(['feature']) -> ok
    [3] get_feature_plot(['feature']) -> ok
    [3] get_feature_plot(['feature']) -> ok
    [3] get_feature_plot(['feature']) -> ok
    [3] get_feature_plot(['feature']) -> ok
    [3] get_feature_plot(['feature']) -> ok
  tooluse_all      XGB  covered=9/9 in=15636 out=

## 3. Split into per-feature records + coverage

`write_split_records` materialises `{condition}_{model}_{feature}.json` so the existing
G3 rubric/judge glob picks them up. The **coverage** table is the direct "how many of 9
features did each whole-model answer actually describe?" measurement. In the stub path we
write to a throwaway temp dir and score with the rubric to prove G3-compatibility — with
`RUN_API=True` the split records are written under `results/global_whole_split/`.

In [8]:
import pandas as pd

def coverage_table(records):
    rows = []
    for rec in records:
        parts = split_whole_model_record(rec, features=FEATURES)
        rows.append({'condition': rec['condition'], 'model': rec['xai_model'],
                     'covered': sum(1 for p in parts if not p['dropped']),
                     'n': N_FEATURES})
    return pd.DataFrame(rows).pivot(index='condition', columns='model', values='covered')

print('Feature coverage (out of 9):')
print(coverage_table(records))

if RUN_API:
    paths = write_split_records(records, features=FEATURES, split_dir=SPLIT_DIR)
    print(f'\nWrote {len(paths)} split records -> {SPLIT_DIR}')
    print('Next: score them in 05Gb (rubric + reference judge, src_subdir="global_whole_split").')
else:
    # verify splitter + rubric end-to-end without touching results/
    with tempfile.TemporaryDirectory() as td:
        paths = write_split_records(records, features=FEATURES, split_dir=td)
        scored = [rubric.score_result_file(Path(p)) for p in paths
                  if json.loads(Path(p).read_text())['explanation']]
        sdf = pd.DataFrame(scored)
        print(f'\n[stub] wrote+scored {len(paths)} split records in a temp dir '
              f'({len(scored)} non-empty).')
        print('[stub] mean rubric total by condition (sanity — correct ranks/direction):')
        print(sdf.groupby('form_pipeline')['total'].mean().round(3))
    print('\nNon-API path verified. Set RUN_API=True for the billed run.')

Feature coverage (out of 9):
model            ebm  xgb
condition                
json_all           9    9
json_beeswarm      9    9
tooluse_all        9    9
vision_all         9    6
vision_beeswarm    9    4

Wrote 90 split records -> /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation-XAI-Stahl-ss26/results/global_whole_split
Next: score them in 05Gb (rubric + reference judge, src_subdir="global_whole_split").
